# Worklet 9 — Placement Readiness & Career Intelligence Portal
## Lab 06: Checkpoint, resume and budgets

**Team:** Vijay Kumar — 01FE23BCI083; Rishi Kulkarni — 01FE23BCI088; Prajwal Hugar — 01FE23BCI075; Rohit Yendegeri — 01FE23BCI001.

**Problem statement:** Help students understand resume/JD gaps and prepare with auditable, placement-officer-reviewed guidance.

| Item | This lab |
| --- | --- |
| Goal | Recover after an injected failure without repeating successful stages. |
| Inputs | A temporary checkpoint file, six-action budget and officer gate. |
| Deliverable | Executable demonstration, assertions and recorded output |
| Acceptance | A new runtime resumes saved work; action budgets, dependencies and approval are enforced. |

Run top to bottom from this repository. The notebook imports `lab_support.py` and existing backend modules; it is independent of other notebooks, but is not a standalone Colab upload. All examples use synthetic data.


### 1. Environment and scope

Only local code and disposable fixtures are used. Dependencies: Python 3.11+ and the repository backend requirements. No secrets, real student records, or reference-notebook outputs are copied.


In [1]:
from pathlib import Path
import sys
folder=Path.cwd()
if not (folder/'lab_support.py').exists():folder=folder/'Assignments'
assert (folder/'lab_support.py').exists(), 'Run from the repository root or Assignments directory'
sys.path.insert(0,str(folder.resolve()))
from lab_support import *
print('Worklet 9: offline synthetic demonstration; no Gemini API calls')


Worklet 9: offline synthetic demonstration; no Gemini API calls


### 2. Worklet-specific implementation and experiment

The following code executes the stated scenario and deliberately exercises rejection paths where relevant.


In [2]:
import tempfile
with tempfile.TemporaryDirectory() as directory:
 path=Path(directory)/'checkpoint.json';runtime=Runtime(path)
 runtime.perform('intake',[],lambda:{'student':'DEMO001'})
 def crash():raise ValueError('Injected connector outage')
 expect(ValueError,lambda:runtime.perform('analysis',['intake'],crash))
 restored=Runtime(path)
 def forbidden():raise AssertionError('Completed intake reran')
 restored.perform('intake',[],forbidden)
 restored.perform('analysis',['intake'],lambda:{'coverage':50})
 expect(PermissionError,lambda:restored.perform('publish',['analysis'],lambda:{'approved':True},human_gate=True))
 restored.perform('publish',['analysis'],lambda:{'approved':True},officer=True,human_gate=True)
 expect(ValueError,lambda:restored.perform('bad',['missing'],lambda:{}))
 limited=Runtime(Path(directory)/'limited.json',budget=0)
 expect(ValueError,lambda:limited.perform('intake',[],lambda:{}))
 show(restored.state)


Expected refusal: Injected connector outage
Expected refusal: Named officer approval required
Expected refusal: Dependency not complete
Expected refusal: Action budget exhausted
{
  "outputs": {
    "intake": {
      "student": "DEMO001"
    },
    "analysis": {
      "coverage": 50
    },
    "publish": {
      "approved": true
    }
  },
  "audit": [
    {
      "stage": "intake",
      "status": "complete",
      "dependencies": [],
      "output": {
        "student": "DEMO001"
      }
    },
    {
      "stage": "analysis",
      "status": "failed",
      "error": "Injected connector outage"
    },
    {
      "stage": "analysis",
      "status": "complete",
      "dependencies": [
        "intake"
      ],
      "output": {
        "coverage": 50
      }
    },
    {
      "stage": "publish",
      "status": "complete",
      "dependencies": [
        "analysis"
      ],
      "output": {
        "approved": true
      }
    }
  ],
  "attempts": 4
}


### 3. Acceptance checks against the actual portal

The demonstration asserts its own contract above. This additional test calls the project implementation using disposable fixtures. Implementation references: `backend/runtime_service.py`.


In [3]:
regression_result=regressions('test_runtime.RuntimeTests.test_checkpoint_resume_and_idempotent_publication')
show(regression_result)


.
----------------------------------------------------------------------
Ran 1 test in 2.064s

OK

{
  "tests": 1,
  "failures": 0,
  "errors": 0
}


### 4. Results, reflection and limits

The teaching runtime logs inputs/dependencies and results to a temporary file. The portal regression verifies real persisted stage reuse and no duplicate draft publication. This is not a proprietary OpenClaw integration.

The output above is generated by execution, not an expected-output transcription. If any assertion or regression fails, this notebook is not accepted.

### 5. Deliverables and next step

- This executed notebook and its assertion output.
- Shared original teaching helpers in `lab_support.py`.
- Relevant production code and regression mapping above.
- Full sequence and provenance in `Assignments/README.md`.

Continue to Lab 7 to build on this contract.
